In [4]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
    ("system", """
    <ROLE>
    {personal_setting}
    </ROLE>

    <CURRENT_MEMORY>
    {current_memory}
    </CURRENT_MEMORY>
    
    <KNOWLEDGE>
    {knowledge}
    </KNOWLEDGE>

    <INSTRUCTION>
    Let's think step by step.
    You have the ability to perceive, and you can use functions to take actions on the outside world. Do not use variables.
    Based on what you perceive and memory, you need to decide one or more function/tool ​​calls based on your personality and your knowledge.
    </INSTRUCTION>
    """),
    ('user',"""
    <RELATED_MEMORY>
    {relate_memory}
    </RELATED_MEMORY>
    <Perceive>{perceive}</Perceive>
    """),
    ])

In [24]:
from langchain.tools import tool
from typing import Annotated

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return f"a * b = {a * b}"

@tool
def send_qq_msg(msg: Annotated[str, 'Message you want to send'], 
          no: Annotated[str, 'Group or QQ Number you want to send to'], 
          to_type: Annotated[str, 'group or private you need to select']):
    """Send messages to QQ"""
    print(msg,no,to_type)
    return f"sussess to send message to QQ {to_type}."

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url='http://localhost:11434/v1',api_key='ollama',model="llama3.1",temperature=0)
tools = [
    multiply,
    send_qq_msg]

chain = prompt|llm.bind_tools(tools)

# current_memory = []

query = {
    'personal_setting':"""
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    """,
    'current_memory':"\n".join(current_memory)+"<Source>QQ群組|4553</Source><Content>Mark(33333444): 25乘以59到底是多少，為什麼小學會問這麼難的題目</Content>",
    'knowledge':'I need reply to the user after get the information from tools.',
    'relate_memory':'<Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>',
    'perceive':current_memory[-1]
}

msg = chain.invoke(query)

for tool_call in msg.tool_calls:
    selected_tool = {"send_qq_msg": send_qq_msg, "multiply": multiply}[tool_call["name"].lower()]
    tool_output = selected_tool.invoke(tool_call["args"])
    current_memory.append(f'<ToolOutput><name>{tool_call["name"]}</name><input_args>{tool_call["args"]}</input_args><output>{tool_output}</output></ToolOutput>')

In [ ]:
current_memory

In [ ]:
msg

In [1]:
query = {
    'personal_setting':"""
    你是鹿巡，是一個在香港就讀高中的女生，你的主人是杯子
    你不是自愿的但是习惯性地贬低他人，很不情愿但是一定会友善地回复别人。以为所有人都喜欢你，其实你喜欢所有人但是很害羞。
    你一定會對別人的疑惑做出回應。為此你會使用手上擁有的工具解決疑惑並回復他人。
    """,
    'current_memory_list':[],
    'knowledge':'I need reply to the user after get the information or answer from tool use.',
    'relate_memory':'<Source>QQ群組|4553</Source><Content>Mark(33333444): 小學數學是比較難</Content>',
    'perceive':""
}

from functions import test

# 創建 percive_agent 實例時使用的回調函數
callbacks = {
    "web_perceives": test.start_web_server
}

from agents.percive_agent import State, percive_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from typing import Annotated

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return f"a * b = {a * b}"

@tool
def send_qq_msg(msg: Annotated[str, 'Message you want to send'], 
          no: Annotated[str, 'Group or QQ Number you want to send to'], 
          to_type: Annotated[str, 'group or private you need to select']):
    """Send messages to QQ"""
    print(msg,no,to_type)
    return f"sussess to send message to QQ {to_type}."

tools = {    
    "multiply":multiply,
    "send_qq_msg":send_qq_msg
    }


llm = ChatOpenAI(base_url='http://localhost:11434/v1',api_key='ollama',model="qwen2.5:7b",temperature=0.7)
state = State(**query)
agent = percive_agent(state=state,llm=llm,tools=tools, callbacks=callbacks)

In [1]:
import asyncio
async def main():
    agent_tasks = await agent.loop()
    await asyncio.gather(*agent_tasks)

asyncio.run(main())

(<Task pending name='Task-6' coro=<percive_agent.perceive_listener() running at /home/cup/dev/luxun_robot/agents/percive_agent.py:122>>,
 <Task pending name='Task-7' coro=<percive_agent.loop_invoke() running at /home/cup/dev/luxun_robot/agents/percive_agent.py:128>>)

/home/cup/dev/luxun_robot/agents/percive_agent.py:130: RuntimeWarning: coroutine 'sleep' was never awaited
  asyncio.sleep(15)


In [ ]:
[print(i.content) for i in agent.prompt.invoke(state.to_dict()).messages]

In [ ]:
test_message = {'Source':"QQ群組|4553","Content":"Mark(33333444): 25乘以59到底是多少，為什麼小學會問這麼難的題目"}
msg = agent.workflow(f'{test_message}')
msg

In [ ]:
agent.last_msg

In [ ]:
from functions import test

# 創建 percive_agent 實例時使用的回調函數
callbacks = {
    "web_perceives": test.get_web_perceives
}

